In [1]:
# Install dependencies
!pip install -q torch transformers datasets scikit-learn tqdm

In [2]:
import os
import shutil
import kagglehub

# Download dataset
dataset_path = kagglehub.dataset_download("riyasuryawanshi/cuad-project")
print("Downloaded to:", dataset_path)

# Set up project directory in Kaggle's writable area
project_dir = "/kaggle/working/cuad_project"
os.makedirs(project_dir, exist_ok=True)

# Copy all files into working directory
for item in os.listdir(dataset_path):
    src = os.path.join(dataset_path, item)
    dst = os.path.join(project_dir, item)
    if os.path.isdir(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
    else:
        shutil.copy2(src, dst)

# Change directory into the project folder
%cd {project_dir}
print("Current Working Directory:", os.getcwd())
!ls -la

Downloaded to: /kaggle/input/datasets/riyasuryawanshi/cuad-project
/kaggle/working/cuad_project
Current Working Directory: /kaggle/working/cuad_project
total 53300
drwxr-xr-x 2 root root     4096 Oct  9 08:20 .
drwxr-xr-x 4 root root     4096 Oct  9 08:20 ..
-rw-r--r-- 1 root root     7185 Oct  9 08:19 category_mapping.csv
-rw-r--r-- 1 root root 14301980 Oct  9 08:19 clauses_mapped.jsonl
-rw-r--r-- 1 root root 40128638 Oct  9 08:19 CUAD_v1.json
-rw-r--r-- 1 root root    25702 Oct  9 08:19 dataset.py
-rw-r--r-- 1 root root    11159 Oct  9 08:19 indian_legal_rules.csv
-rw-r--r-- 1 root root     8992 Oct  9 08:19 metrics.py
-rw-r--r-- 1 root root    10476 Oct  9 08:19 model.py
-rw-r--r-- 1 root root    20613 Oct  9 08:19 smoke_test.py
-rw-r--r-- 1 root root    35895 Oct  9 08:19 train_phase3.py


In [3]:
!python smoke_test.py

[00:00:00] Tokenize words                 ██████████████████ 169      /      169[00:00:00] Tokenize words                 ██████████████████ 0        /        0
[00:00:00] Count pairs                    ██████████████████ 169      /      169
[00:00:00] Compute merges                 ██████████████████ 492      /      492
PASS  contract text re-attachment verifies every offset
PASS  span window: always contains the answer exactly, and is NOT always centred
PASS  span labels round-trip: tokens start..end decode back to the clause text
PASS  multi-gold: two same-category spans in one window both counted as valid answers
PASS  span tokens: question/special tokens are global-attention, context is not
PASS  splits are contract-disjoint; a deliberate leak is caught
PASS  forward: output shapes for all three heads
PASS  gradients reach every head and the encoder; loss is finite
PASS  masked losses: unlabeled batch -> no loss; partial labels -> only those tasks
PASS  ordinal risk head: score re

In [ ]:
!python train_phase3.py --experiments cls_only span_only joint_cls_span --epochs 2 --window_chars 3000 --max_len_span 1024 --span_train_frac 0.5 --cv_folds 1 --out_dir /kaggle/working/cuad_project/stageB --resume

device=cuda amp=torch.float16
contract text attached + offsets verified: {'verified': 13823}
9786 examples from 481 contracts; [('Intellectual Property', 1955), ('Termination', 1495), ('Liability', 1399), ('Commercial Terms', 1078), ('Other', 1075), ('Assignment', 907), ('Non-Compete', 794), ('Governing Law / Jurisdiction', 464), ('Warranty', 176), ('Dispute Resolution', 173), ('Non-Solicitation', 149), ('Penalties', 121)]
fold 0: split sizes / category balance / risk-labelled counts |   train  n= 6914 risk_labelled=   0 categories=12 |   val    n=  889 risk_labelled=   0 categories=12 |   test   n= 1983 risk_labelled=   0 categories=12
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/allenai/longformer-base-4096/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/allenai/longformer-base-4096/resolve/main/config.json
Retrying in 2s 

In [5]:
import os

print("Files in stageB:")
if os.path.exists('/kaggle/working/cuad_project/stageB'):
    print(os.listdir('/kaggle/working/cuad_project/stageB'))
else:
    print("stageB folder not found.")

Files in stageB:
['partial', 'results.json', 'results.md']


In [6]:
!python train_phase3.py --experiments cls_only span_only joint_cls_span --epochs 2 --window_chars 3000 --max_len_span 1024 --span_train_frac 0.5 --cv_folds 1 --out_dir /kaggle/working/cuad_project/stageB --resume

device=cuda amp=torch.float16
contract text attached + offsets verified: {'verified': 13823}
9786 examples from 481 contracts; [('Intellectual Property', 1955), ('Termination', 1495), ('Liability', 1399), ('Commercial Terms', 1078), ('Other', 1075), ('Assignment', 907), ('Non-Compete', 794), ('Governing Law / Jurisdiction', 464), ('Warranty', 176), ('Dispute Resolution', 173), ('Non-Solicitation', 149), ('Penalties', 121)]
fold 0: split sizes / category balance / risk-labelled counts |   train  n= 6914 risk_labelled=   0 categories=12 |   val    n=  889 risk_labelled=   0 categories=12 |   test   n= 1983 risk_labelled=   0 categories=12
=== cls_only | fold 1/1 === already done, loading
=== span_only | fold 1/1 === already done, loading
=== joint_cls_span | fold 1/1 === already done, loading

Wrote /kaggle/working/cuad_project/stageB/results.json and /kaggle/working/cuad_project/stageB/results.md
